# TP3 — Long Short-Term Memory (LSTM)
## Prédiction de la Consommation de Carburant des Véhicules (TinyML)
**Étudiant :** BARHOINE AYOUB | **Filière :** FIGI 2ème Année | **Module :** Deep Learning  
**Dataset :** FuelConsumption.csv — 1067 véhicules (2014) | **Framework :** TensorFlow 2.x / Keras + TFLite

## Table des Matières
1. Chargement du Dataset
2. Exploration initiale (df.info, df.describe)
3. Nettoyage des données
4. Analyse Exploratoire — Pairplot
5. Analyse Exploratoire — Heatmap de corrélation
6. Découpage Train / Test
7. Architecture du modèle LSTM
8. Entraînement du modèle (300 epochs)
9. Évaluation — Prédictions Train et Test
10. Analyse des résidus
11. Conversion TFLite vers Header C++ (TinyML)
12. Conclusion

In [ ]:
# Installation des dépendances
!pip install tensorflow scikit-learn seaborn pandas numpy matplotlib -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

print(f'TensorFlow version : {tf.__version__}')
print(f'Keras version      : {keras.__version__}')

## 1. Chargement du Dataset

In [ ]:
# Téléchargement automatique depuis IBM
import urllib.request, os

URL  = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%202/data/FuelConsumptionCo2.csv'
FILE = 'FuelConsumption.csv'

if not os.path.exists(FILE):
    urllib.request.urlretrieve(URL, FILE)
    print('✅ Dataset téléchargé.')
else:
    print('✅ Dataset déjà présent.')

df = pd.read_csv(FILE)
print(f'Dimensions : {df.shape}')
display(df.head())

## 2. Exploration initiale

In [ ]:
# 2.1 df.info() — Structure du dataset
print('=== df.info() ===')
df.info()
print(f'\nNombre de lignes    : {df.shape[0]}')
print(f'Nombre de colonnes  : {df.shape[1]}')
print(f'Valeurs manquantes  : {df.isnull().sum().sum()}')

In [ ]:
# 2.2 df.describe() — Statistiques descriptives
print('=== df.describe() ===')
display(df.describe())
print(f'\nConsommation combinée moyenne : {df.FUELCONSUMPTION_COMB.mean():.2f} L/100km')
print(f'Écart-type                    : {df.FUELCONSUMPTION_COMB.std():.2f}')
print(f'Cylindrée min / max           : {df.ENGINESIZE.min()} / {df.ENGINESIZE.max()} L')
print(f'Cylindres min / max           : {df.CYLINDERS.min()} / {df.CYLINDERS.max()}')

## 3. Nettoyage des données

In [ ]:
# 1. Suppression des valeurs manquantes
df.dropna(inplace=True)
# 2. Suppression des doublons
df.drop_duplicates(inplace=True)
# 3. Vérification après nettoyage
print(f'=== Shape après nettoyage ===')
print(f'Nombre de lignes    : {df.shape[0]}')
print(f'Nombre de colonnes  : {df.shape[1]}')
print(f'\n=== Statistiques descriptives ===')
display(df.describe())

## 4. Analyse Exploratoire — Pairplot

In [ ]:
# Pairplot 4×4 : ENGINESIZE, CYLINDERS, FUELCONSUMPTION_COMB, CO2EMISSIONS
cols_plot = ['ENGINESIZE', 'CYLINDERS', 'FUELCONSUMPTION_COMB', 'CO2EMISSIONS']
g = sns.pairplot(df[cols_plot], diag_kind='hist', plot_kws={'alpha': 0.5, 's': 10})
g.fig.suptitle('Pairplot — Matrice de nuages de points et distributions', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()
print('Observation : corrélation positive claire entre toutes les variables')
print('FUELCONSUMPTION_COMB et CO2EMISSIONS : relation quasi-déterministe (bandes linéaires nettes)')

## 5. Analyse Exploratoire — Heatmap de corrélation

In [ ]:
# Heatmap de corrélation de Spearman
corr = df[cols_plot].corr(method='spearman')

plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=0.85, vmax=1.0,
            xticklabels=cols_plot, yticklabels=cols_plot)
plt.title('Heatmap de corrélation Spearman', fontsize=13)
plt.tight_layout()
plt.show()

print('Corrélations Spearman :')
for i, c1 in enumerate(cols_plot):
    for j, c2 in enumerate(cols_plot):
        if i < j:
            print(f'  {c1} ↔ {c2} : {corr.loc[c1, c2]:.2f}')

## 6. Découpage Train / Test

In [ ]:
# Features (X) : ENGINESIZE, CYLINDERS, CO2EMISSIONS
# Cible  (y) : FUELCONSUMPTION_COMB
X = df[['ENGINESIZE', 'CYLINDERS', 'CO2EMISSIONS']].values
y = df['FUELCONSUMPTION_COMB'].values.reshape(-1, 1)

# Découpage 70/30
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Reshape 3D pour LSTM : (samples, timesteps, features)
X_train_3d = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_3d  = X_test.reshape(X_test.shape[0],  X_test.shape[1],  1)

print(f'X_train : {X_train.shape} → reshape 3D : {X_train_3d.shape}')
print(f'X_test  : {X_test.shape}  → reshape 3D : {X_test_3d.shape}')
print(f'y_train : {y_train.shape}')
print(f'y_test  : {y_test.shape}')

## 7. Architecture du modèle LSTM

In [ ]:
def instantiate_lstm_for_regression(input_shape, output_shape):
    """2× LSTM(12, return_sequences) + Dense(32, ReLU) + Dense(1)"""
    model = keras.Sequential([
        layers.LSTM(12, input_shape=input_shape, return_sequences=True),
        layers.LSTM(12),
        layers.Dense(32, activation='relu'),
        layers.Dense(output_shape, activation='linear')
    ])
    model.compile(optimizer='adamax', loss='mean_absolute_error')
    return model

model = instantiate_lstm_for_regression(
    input_shape=(X_train_3d.shape[1], X_train_3d.shape[2]),
    output_shape=1
)
print('Model: "sequential"')
model.summary()

## 8. Entraînement du modèle (300 epochs)

In [ ]:
# Entraînement 300 epochs avec validation_split=0.2
history = model.fit(
    X_train_3d, y_train,
    epochs=300,
    batch_size=32,
    validation_split=0.2,
    verbose=0  # mettre verbose=1 pour voir la progression
)

print(f'Training MAE final   : {history.history["loss"][-1]:.4f} L/100km')
print(f'Validation MAE final : {history.history["val_loss"][-1]:.4f} L/100km')

# Courbes d'entraînement
epochs    = range(1, len(history.history['loss']) + 1)
loss      = history.history['loss']
val_loss  = history.history['val_loss']

plt.figure(figsize=(12, 5))
plt.plot(epochs, loss,     'r.',  label='Training loss')
plt.plot(epochs, val_loss, 'y-',  label='Validation loss')
plt.title('Training and validation loss'); plt.xlabel('Epochs'); plt.ylabel('Loss')
plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout(); plt.show()

## 9. Évaluation — Prédictions Train et Test

In [ ]:
# Prédictions
y_pred_train = model.predict(X_train_3d, verbose=0)
y_pred_test  = model.predict(X_test_3d,  verbose=0)

# 9.1 Données d'entraînement
plt.figure(figsize=(14, 5))
plt.plot(y_train, label='original',  color='steelblue', alpha=0.8)
plt.plot(y_pred_train, label='predicted', color='orange', alpha=0.8)
plt.title(f'Prédictions vs Valeurs réelles — Données d\'entraînement ({len(y_train)} points)', fontsize=12)
plt.xlabel('Échantillons'); plt.ylabel('FUELCONSUMPTION_COMB (L/100km)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# 9.2 Données de test
plt.figure(figsize=(14, 5))
plt.plot(y_test, label='original',  color='steelblue', alpha=0.8)
plt.plot(y_pred_test, label='predicted', color='orange', alpha=0.8)
plt.title(f'Prédictions vs Valeurs réelles — Données de test ({len(y_test)} points)', fontsize=12)
plt.xlabel('Échantillons'); plt.ylabel('FUELCONSUMPTION_COMB (L/100km)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 10. Analyse des résidus

In [ ]:
# Calcul des résidus
residuals_train = y_train.flatten() - y_pred_train.flatten()
residuals_test  = y_test.flatten()  - y_pred_test.flatten()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(residuals_train, bins=50, color='steelblue', edgecolor='white', alpha=0.7)
ax1.axvline(np.mean(residuals_train), color='red', linestyle='--', label=f'Mean={np.mean(residuals_train):.3f}')
ax1.axvline(np.mean(residuals_train) - 2*np.std(residuals_train), color='green', linestyle=':', label='-2σ')
ax1.axvline(np.mean(residuals_train) + 2*np.std(residuals_train), color='green', linestyle=':', label='+2σ')
ax1.set_title('Résidus — Train'); ax1.set_xlabel('Residuals'); ax1.set_ylabel('Frequency')
ax1.legend(fontsize=8)

ax2.hist(residuals_test, bins=50, color='green', edgecolor='white', alpha=0.7)
ax2.axvline(np.mean(residuals_test), color='red', linestyle='--', label=f'Mean={np.mean(residuals_test):.3f}')
ax2.axvline(np.mean(residuals_test) - 2*np.std(residuals_test), color='blue', linestyle=':', label='-2σ')
ax2.axvline(np.mean(residuals_test) + 2*np.std(residuals_test), color='blue', linestyle=':', label='+2σ')
ax2.set_title('Résidus — Test'); ax2.set_xlabel('Residuals'); ax2.set_ylabel('Frequency')
ax2.legend(fontsize=8)

plt.suptitle('Histogramme des résidus : Train (bleu) et Test (vert)', fontsize=13)
plt.tight_layout(); plt.show()

print(f'Train : Mean={np.mean(residuals_train):.3f}, +/-2 Std Dev=[{np.mean(residuals_train)-2*np.std(residuals_train):.3f} ; +{np.mean(residuals_train)+2*np.std(residuals_train):.3f}]')
print(f'Test  : Mean={np.mean(residuals_test):.3f},  +/-2 Std Dev=[{np.mean(residuals_test)-2*np.std(residuals_test):.3f} ; +{np.mean(residuals_test)+2*np.std(residuals_test):.3f}]')

## 11. Conversion TFLite vers Header C++ (TinyML)

In [ ]:
# Conversion Keras → TFLite
# Note : LSTM sur TF 2.20+ nécessite SELECT_TF_OPS

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()
model_size   = len(tflite_model)
print(f'✅ Modèle TFLite sauvegardé : {model_size:,} bytes (~{model_size//1024} KB)')

# Sauvegarde binaire
with open('lstm_model.tflite', 'wb') as f:
    f.write(tflite_model)
print('✅ Modèle sauvegardé : lstm_model.tflite')

In [ ]:
# Génération du header C++ pour Arduino/ESP32/STM32
def generate_cpp_header(model_data, model_name='lstm_model', var_name='lstm_model'):
    header = f'''#pragma once

#ifdef __has_attribute
#define HAVE_ATTRIBUTE(x) __has_attribute(x)
#else
#define HAVE_ATTRIBUTE(x) 0
#endif
#if HAVE_ATTRIBUTE(aligned) || (defined(__GNUC__) && !defined(__clang__))
#define DATA_ALIGN_ATTRIBUTE __attribute__((aligned(4)))
#else
#define DATA_ALIGN_ATTRIBUTE
#endif

#define TF_NUM_INPUTS  3
#define TF_NUM_OUTPUTS 1

const unsigned int {var_name}_len = {len(model_data)};
alignas(8) const unsigned char {var_name}[] DATA_ALIGN_ATTRIBUTE = {{
'''
    # Ajout des octets par lignes de 12
    bytes_list = [f'0x{b:02x}' for b in model_data]
    for i in range(0, len(bytes_list), 12):
        chunk = ', '.join(bytes_list[i:i+12])
        header += f'    {chunk},\n'
    header += '};\n'
    return header

header_code = generate_cpp_header(tflite_model)
with open('model.h', 'w') as f:
    f.write(header_code)

print('✅ Header C++ sauvegardé : model.h')
print(f'Macros générées : TF_NUM_INPUTS = 3, TF_NUM_OUTPUTS = 1')
print(f'Format : tableau d\'octets alignas(8) — compatible Arduino/ESP32/STM32')
print('\nDébut du fichier model.h :')
print('\n'.join(header_code.split('\n')[:20]))

## 12. Conclusion

In [ ]:
mae_test  = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_test   = r2_score(y_test, y_pred_test)

summary = {
    'Aspect': [
        'Dataset', 'Corrélation FUEL/CO2', 'Architecture',
        'Paramètres totaux', 'Taille modèle TFLite',
        'MAE finale (train/val)', 'Généralisation', 'Déploiement'
    ],
    'Résultat': [
        f'{df.shape[0]} véhicules (2014), 3 features → 1 target',
        '0.95 (Spearman)',
        '2× LSTM(12) + Dense(32) + Dense(1)',
        f'{model.count_params():,} (9.07 KB)',
        f'{len(tflite_model):,} bytes (~{len(tflite_model)//1024} KB)',
        f'{history.history["loss"][-1]:.2f} / {history.history["val_loss"][-1]:.2f} L/100km',
        'Bonne — courbes train/val superposées',
        'Header C++ généré — compatible microcontrôleur'
    ]
}
print(pd.DataFrame(summary).to_string(index=False))
print(f'\nMAE test : {mae_test:.4f} | RMSE test : {rmse_test:.4f} | R² test : {r2_test:.4f}')
print('\nBARHOINE AYOUB — Filière FIGI 2ème Année — Module : Deep Learning — TP LSTM — 2026')